# Chapter 14: Simulation and Evaluation

<a href="../lite/lab/index.html?path=ch14_simulation_evaluation.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    e = Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(e)

Before you test on a real robot, test on a fake one. Not because simulation is easier (it is),
but because simulation gives you something the real world never does: the **ground truth**.
When your filter says the robot is at $(3.2, 1.7)$, simulation lets you check whether it is
actually at $(3.2, 1.7)$ or at $(5.1, -0.3)$.

This chapter builds the simulation testbed that we will use for all remaining chapters.
It also introduces the metrics for evaluating how well an estimator performs.

## 14.1 Synthetic Environments

A good simulation needs:
1. A **world** with features (walls, landmarks)
2. A **robot** with a motion model
3. **Sensors** with realistic noise models
4. **Ground truth** recorded at every step

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
room_width = 20.0       # room width (meters)
room_height = 15.0      # room height (meters)
n_landmarks = 8         # number of point landmarks
seed = 42
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(seed)

# Room walls
walls = np.array([
    [0, 0, room_width, 0],           # bottom
    [room_width, 0, room_width, room_height],  # right
    [room_width, room_height, 0, room_height],  # top
    [0, room_height, 0, 0]            # left
])

# Random landmarks inside the room
landmarks = np.column_stack([
    np.random.uniform(1, room_width - 1, n_landmarks),
    np.random.uniform(1, room_height - 1, n_landmarks)
])

# Robot trajectory: a loop
t = np.linspace(0, 2*np.pi, 200)
cx, cy = room_width/2, room_height/2
rx, ry = room_width/3, room_height/3
traj_x = cx + rx * np.cos(t)
traj_y = cy + ry * np.sin(t)
traj_theta = np.arctan2(np.gradient(traj_y), np.gradient(traj_x))

fig, ax = plt.subplots(figsize=(12, 8))

# Draw walls
for w in walls:
    ax.plot([w[0], w[2]], [w[1], w[3]], 'k-', lw=3)

# Draw landmarks
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=120, marker='^', zorder=5, label='landmarks')
for i, lm in enumerate(landmarks):
    ax.annotate(f'L{i}', xy=lm, xytext=(5, 5), textcoords='offset points', fontsize=10, color='steelblue')

# Draw trajectory
ax.plot(traj_x, traj_y, 'tomato', lw=2, label='ground truth trajectory')
ax.plot(traj_x[0], traj_y[0], 'go', ms=12, zorder=5, label='start')
# Draw robot every 20 steps
for i in range(0, len(t), 40):
    dx = 0.8 * np.cos(traj_theta[i])
    dy = 0.8 * np.sin(traj_theta[i])
    ax.annotate("", xy=(traj_x[i]+dx, traj_y[i]+dy), xytext=(traj_x[i], traj_y[i]),
                arrowprops=dict(arrowstyle='->', color='tomato', lw=1.5))

ax.set_xlim(-1, room_width+1); ax.set_ylim(-1, room_height+1)
ax.set_aspect('equal')
ax.set_title("Synthetic environment: room with landmarks and robot trajectory", fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 14.2 Noise Modeling

A simulation is only useful if its noise is realistic. Too little noise and your algorithm
looks better than it really is. Too much and you are solving a harder problem than reality.

Common noise models:
- **Odometry**: Gaussian noise proportional to distance traveled
- **Range sensor**: Gaussian noise + occasional outliers
- **Bearing sensor**: Gaussian noise, worse at long range

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
sigma_odom_x = 0.1       # odometry noise in x (m per step)
sigma_odom_y = 0.05      # odometry noise in y (m per step)
sigma_odom_theta = 0.02  # odometry noise in heading (rad per step)
sigma_range = 0.3        # range measurement noise (m)
sigma_bearing = 0.05     # bearing measurement noise (rad)
outlier_prob = 0.05      # probability of range outlier
max_sensor_range = 8.0   # maximum sensor range (m)
n_steps = 200
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Generate noisy odometry
odom_x = traj_x + np.cumsum(np.random.normal(0, sigma_odom_x, n_steps))
odom_y = traj_y + np.cumsum(np.random.normal(0, sigma_odom_y, n_steps))

# Generate range-bearing measurements at step 50
robot_pos = np.array([traj_x[50], traj_y[50]])
robot_theta = traj_theta[50]

ranges_true = np.linalg.norm(landmarks - robot_pos, axis=1)
bearings_true = np.arctan2(landmarks[:, 1] - robot_pos[1], landmarks[:, 0] - robot_pos[0]) - robot_theta

# Add noise and outliers
ranges_noisy = ranges_true + np.random.normal(0, sigma_range, n_landmarks)
bearings_noisy = bearings_true + np.random.normal(0, sigma_bearing, n_landmarks)
# Add outliers
for i in range(n_landmarks):
    if np.random.random() < outlier_prob:
        ranges_noisy[i] = np.random.uniform(0, max_sensor_range)

visible = ranges_true < max_sensor_range

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.plot(traj_x, traj_y, 'steelblue', lw=2, label='ground truth')
ax.plot(odom_x, odom_y, 'tomato', lw=1, alpha=0.7, label='odometry (noisy)')
ax.set_title("Odometry drift", fontsize=13); ax.set_aspect('equal'); ax.legend()

ax = axes[1]
ax.bar(range(n_landmarks), ranges_true, alpha=0.5, color='steelblue', label='true range')
ax.bar(range(n_landmarks), ranges_noisy, alpha=0.5, color='tomato', label='noisy range')
ax.set_xlabel("Landmark"); ax.set_ylabel("Range (m)"); ax.set_title("Range measurements", fontsize=13); ax.legend()

ax = axes[2]
ax.bar(range(n_landmarks), np.degrees(bearings_true), alpha=0.5, color='steelblue', label='true bearing')
ax.bar(range(n_landmarks), np.degrees(bearings_noisy), alpha=0.5, color='tomato', label='noisy bearing')
ax.set_xlabel("Landmark"); ax.set_ylabel("Bearing (deg)"); ax.set_title("Bearing measurements", fontsize=13); ax.legend()

plt.tight_layout()
plt.show()

## 14.3 Metrics

How do we measure the quality of an estimator? Common metrics:

| Metric | Formula | Measures |
|--------|---------|----------|
| **ATE** (Absolute Trajectory Error) | $\sqrt{\frac{1}{N}\sum_t \|\hat{x}_t - x_t\|^2}$ | Overall accuracy |
| **RPE** (Relative Pose Error) | Error in relative transforms between consecutive poses | Local consistency |
| **NEES** (Normalized Estimation Error Squared) | $(x - \hat{x})^T P^{-1} (x - \hat{x})$ | Filter consistency |
| **ANEES** | Average NEES over time | Should equal state dimension if consistent |

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
estimator_bias = 0.5      # systematic bias in estimator (try 0, 0.5, 2.0)
estimator_noise = 0.3     # random noise in estimator
# ──────────────────────────────────────────────────────────────────────────────

# Ground truth
gt = np.column_stack([traj_x, traj_y])

# Simulated estimator output (ground truth + bias + noise)
estimated = gt + estimator_bias + np.random.normal(0, estimator_noise, gt.shape)

# Odometry (drifts over time)
odom = np.column_stack([odom_x, odom_y])

# ATE
ate_est = np.sqrt(np.mean(np.sum((estimated - gt)**2, axis=1)))
ate_odom = np.sqrt(np.mean(np.sum((odom - gt)**2, axis=1)))

# RPE (relative pose error over 1 step)
gt_rel = np.diff(gt, axis=0)
est_rel = np.diff(estimated, axis=0)
odom_rel = np.diff(odom, axis=0)
rpe_est = np.sqrt(np.mean(np.sum((est_rel - gt_rel)**2, axis=1)))
rpe_odom = np.sqrt(np.mean(np.sum((odom_rel - gt_rel)**2, axis=1)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(gt[:, 0], gt[:, 1], 'k-', lw=2, label='ground truth')
ax.plot(estimated[:, 0], estimated[:, 1], 'steelblue', lw=1.5, label=f'estimator (ATE={ate_est:.3f}m)')
ax.plot(odom[:, 0], odom[:, 1], 'tomato', lw=1, alpha=0.6, label=f'odometry (ATE={ate_odom:.3f}m)')
ax.set_title("Trajectory comparison", fontsize=13); ax.set_aspect('equal'); ax.legend(fontsize=9)

ax = axes[1]
errors_est = np.sqrt(np.sum((estimated - gt)**2, axis=1))
errors_odom = np.sqrt(np.sum((odom - gt)**2, axis=1))
ax.plot(errors_est, 'steelblue', lw=2, label='estimator error')
ax.plot(errors_odom, 'tomato', lw=1.5, label='odometry error')
ax.set_xlabel("Time step"); ax.set_ylabel("Position error (m)")
ax.set_title("Error over time", fontsize=13); ax.legend()

plt.tight_layout()
plt.show()

print(f"{'Metric':<25} {'Estimator':>12} {'Odometry':>12}")
print(f"{'ATE (m)':<25} {ate_est:>12.4f} {ate_odom:>12.4f}")
print(f"{'RPE (m/step)':<25} {rpe_est:>12.4f} {rpe_odom:>12.4f}")

## 14.4 Failure Design

Good simulations test not just normal operation but **failure cases**:
- Sensor dropout (no measurements for several steps)
- Outlier bursts (many wrong readings at once)
- Wrong initial estimate (kidnapped robot)
- Symmetrical environments (perceptual aliasing)

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
dropout_start = 80       # step where sensor drops out
dropout_duration = 40    # number of steps without measurements
# ──────────────────────────────────────────────────────────────────────────────

# Simulate position error during dropout: uncertainty grows linearly
errors_normal = np.ones(n_steps) * 0.3  # normal error
errors_dropout = errors_normal.copy()
for i in range(dropout_start, min(dropout_start + dropout_duration, n_steps)):
    errors_dropout[i] = 0.3 + 0.1 * (i - dropout_start)  # growing error
# After dropout, measurements return and error recovers
for i in range(dropout_start + dropout_duration, n_steps):
    errors_dropout[i] = max(0.3, errors_dropout[i-1] * 0.9)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(errors_normal, 'steelblue', lw=2, label='normal operation')
ax.plot(errors_dropout, 'tomato', lw=2, label='with sensor dropout')
ax.axvspan(dropout_start, dropout_start + dropout_duration, alpha=0.15, color='red', label='dropout period')
ax.set_xlabel("Time step"); ax.set_ylabel("Position error (m)")
ax.set_title("Testing failure resilience: sensor dropout", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 14.5 Visualization

Clear visualization is essential for debugging SLAM systems. Common plots:
- Trajectory overlay (estimated vs ground truth)
- Error time series
- Covariance ellipses over time
- Map quality comparison

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_poses_to_show = 10
sigma_est = 0.5
# ──────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 8))

# Draw room
for w in walls:
    ax.plot([w[0], w[2]], [w[1], w[3]], 'k-', lw=3)

# Ground truth trajectory
ax.plot(traj_x, traj_y, 'k--', lw=1, alpha=0.5, label='ground truth')

# Estimated trajectory with covariance ellipses
step_indices = np.linspace(0, n_steps-1, n_poses_to_show, dtype=int)
for i in step_indices:
    cov = np.eye(2) * sigma_est * (1 + 0.01*i)  # growing uncertainty
    draw_cov_ellipse(ax, [estimated[i, 0], estimated[i, 1]], cov,
                     fill=True, facecolor='steelblue', alpha=0.15, edgecolor='steelblue', lw=1)
ax.plot(estimated[:, 0], estimated[:, 1], 'steelblue', lw=2, label='estimated')

# Landmarks
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='tomato', s=100, marker='^', zorder=5, label='landmarks')

ax.set_xlim(-1, room_width+1); ax.set_ylim(-1, room_height+1)
ax.set_aspect('equal')
ax.set_title("Full SLAM visualization: trajectory + uncertainty + landmarks", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

**Key observations:**
- **Simulation is not cheating.** It is the only way to systematically test edge cases.
- **ATE** measures overall accuracy. **RPE** measures local consistency. Use both.
- Always test failure cases: dropouts, outliers, wrong initialization.
- The simulation testbed you build here will be reused in every chapter from here on.

---

## Exercises

### Exercise 14.1: Build your own testbed

Create a simulation with: a 30x20m room, 12 landmarks, a figure-8 trajectory (200 steps).
Generate noisy odometry and range-bearing measurements. Compute ATE for odometry alone.

In [ ]:
# Your code here
# 1. Define room walls
# 2. Place landmarks randomly
# 3. Generate figure-8 trajectory
# 4. Add odometry noise
# 5. Generate range-bearing measurements with noise
# 6. Compute ATE

### Exercise 14.2: NEES consistency check

Simulate a 1D Kalman filter tracking position with known dynamics.
Compute the NEES at each step: $(x - \hat{x})^2 / P$.
If the filter is consistent, the average NEES should be approximately 1.0.
Try making the process noise wrong (too low) and show that NEES increases.

In [ ]:
# Your code here
# Hint: run the KF from ch16 and compute NEES at each step

### Exercise 14.3: Monte Carlo evaluation

Run your estimator 100 times with different random seeds.
Plot the distribution of ATE values. Report the mean and 95th percentile.
This gives you a statistically meaningful performance assessment.

In [ ]:
# Your code here
# for seed in range(100):
#     np.random.seed(seed)
#     # run simulation
#     # compute ATE
# Plot histogram of ATEs

### Exercise 14.4: Comprehensive testbed (challenge)

Build the complete testbed that will be used in chapters 15 onward:
- Square room with 8 landmarks
- Robot drives a rectangular loop (4 straight segments, 4 turns)
- Odometry with realistic noise
- Range-bearing sensor with max range, noise, and 5% outliers
- Record everything: ground truth poses, odometry, measurements, true associations
- Store in a dictionary for easy reuse

In [ ]:
# Your code here
# This is your SLAM testbed. Make it good!
# testbed = {
#     'landmarks': ...,
#     'gt_poses': ...,
#     'odometry': ...,
#     'measurements': [...],  # list of (step, landmark_id, range, bearing)
#     'params': {'sigma_odom': ..., 'sigma_range': ..., ...}
# }